# Anotador TDAH · 05 · Comparación de experimentos

Este cuaderno **no llama al modelo**: lee la tabla `experimento` y compara los experimentos guardados por los cuadernos 01–04 usando sus códigos.

Pregunta que responde: **¿qué configuración (backend, temperatura, modelo...) da la anotación más estable, a qué coste?**

## 1 · Parámetros

In [ ]:
import json
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

RUTA_BD = "datos/anotador.db"
CODIGOS = None   # None = todos los experimentos; o lista de códigos:
                 # ["directo-s1-t0.7-20260712", "langchain-s1-t0.7-20260712"]

con = sqlite3.connect(RUTA_BD)

## 2 · Experimentos disponibles

In [ ]:
df = pd.read_sql("SELECT * FROM experimento", con)
if CODIGOS:
    df = df[df["codigo"].isin(CODIGOS)]

print(f"{len(df)} anotaciones de {df['codigo'].nunique()} experimentos\n")
df.groupby("codigo").agg(
    backend=("backend", "first"),
    modelo=("modelo", "first"),
    temperatura=("temperatura", "first"),
    semana=("semana", "first"),
    anotaciones=("id", "count"),
    primera=("creada_en", "min"),
)

## 3 · Tabla comparativa

Una fila por experimento:
- `formato_ok`: fracción de salidas con JSON válido.
- `acuerdo_nivel`: para cada entrada, fracción de repeticiones que coincide con el nivel más frecuente; luego la media del experimento. 1.0 = totalmente estable.
- `latencia_media`: segundos por anotación.

In [ ]:
def acuerdo_modal(niveles):
    '''Fracción de repeticiones que coincide con el nivel más frecuente.'''
    s = niveles.dropna()
    return s.value_counts().iloc[0] / len(s) if len(s) else None


por_entrada = (
    df.groupby(["codigo", "id_entrada"])["nivel_alerta"]
    .apply(acuerdo_modal)
    .rename("acuerdo_nivel")
)

comparativa = (
    df.groupby("codigo")
    .agg(
        backend=("backend", "first"),
        temperatura=("temperatura", "first"),
        semana=("semana", "first"),
        formato_ok=("formato_ok", "mean"),
        latencia_media=("latencia_s", "mean"),
    )
    .join(por_entrada.groupby("codigo").mean())
    .sort_values("acuerdo_nivel", ascending=False)
    .round(2)
)
comparativa

## 4 · Gráfico: estabilidad frente a coste

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

comparativa["acuerdo_nivel"].plot.barh(ax=ax[0], color="tab:blue")
ax[0].set_xlim(0, 1.05)
ax[0].set_xlabel("acuerdo del nivel de alerta (1.0 = estable)")
ax[0].set_title("Estabilidad por experimento")
ax[0].grid(alpha=0.3, axis="x")

comparativa["latencia_media"].plot.barh(ax=ax[1], color="tab:red")
ax[1].set_xlabel("latencia media (s)")
ax[1].set_title("Coste por experimento")
ax[1].grid(alpha=0.3, axis="x")

plt.tight_layout(); plt.show()

## 5 · Detalle: dónde discrepan las repeticiones

Para un experimento concreto, los pacientes donde el modelo cambia de opinión entre repeticiones (los casos interesantes de revisar a mano).

In [ ]:
UN_EXPERIMENTO = df["codigo"].iloc[0]   # cambiar por el que interese

d = df[df["codigo"] == UN_EXPERIMENTO]
inestables = (
    d.groupby("id_paciente")["nivel_alerta"]
    .apply(lambda s: s.nunique() > 1)
)
print(f"Experimento: {UN_EXPERIMENTO}")
print(f"Pacientes con nivel inestable: {inestables.sum()} de {len(inestables)}\n")

for paciente in inestables[inestables].index:
    filas = d[d["id_paciente"] == paciente]
    niveles = " / ".join(str(x) for x in filas["nivel_alerta"])
    print(f"  {paciente}: {niveles}")

## 6 · Conclusión

Para rellenar tras comparar:

1. **Configuración recomendada**: la de mayor `acuerdo_nivel` con `formato_ok` ≈ 1.
2. **Compromiso estabilidad/coste**: ¿el mejor justifica su latencia frente al segundo?
3. **Casos inestables**: revisar los textos de los pacientes donde el modelo discrepa consigo mismo — suelen ser reportes ambiguos.